# C12-classical-models — Practice p20 — Solution


Each seed receives exactly one random initialization. Co-clustering matrices compare partitions without depending on arbitrary cluster numbers.


In [ ]:
import numpy as np
from sklearn.cluster import KMeans

rng_p20 = np.random.default_rng(20260804)
X_p20 = np.vstack([
    rng_p20.normal(loc=(-3.0, 0.0), scale=0.55, size=(30, 2)),
    rng_p20.normal(loc=(0.0, 3.0), scale=0.55, size=(30, 2)),
    rng_p20.normal(loc=(3.0, 0.0), scale=0.55, size=(30, 2)),
]).astype(np.float64)


def kmeans_stability_audit(X):
    seeds = np.arange(20260804, 20260812, dtype=np.int64)
    models = [KMeans(n_clusters=3, init="random", n_init=1, max_iter=100, tol=1e-6,
                     random_state=int(seed)).fit(X) for seed in seeds]
    inertias = np.array([model.inertia_ for model in models], dtype=np.float64)
    iterations = np.array([model.n_iter_ for model in models], dtype=np.int64)
    labels = np.vstack([model.labels_ for model in models]).astype(np.int64)
    centers = np.stack([model.cluster_centers_ for model in models]).astype(np.float64)
    coclustering = labels[:, :, None] == labels[:, None, :]
    best_index = int(np.argmin(inertias))
    upper = np.triu_indices(X.shape[0], k=1)
    agreements = np.array([np.mean(matrix[upper] == coclustering[best_index][upper])
                           for matrix in coclustering], dtype=np.float64)
    return {"seeds": seeds, "models": tuple(models), "inertias": inertias, "iterations": iterations,
            "labels": labels, "centers": centers, "co_clustering": coclustering,
            "best_index": best_index, "best_seed": int(seeds[best_index]),
            "agreements_to_best": agreements}


audit_p20 = kmeans_stability_audit(X_p20)
interpretation_p20 = '''Inertia answers only the within-cluster squared-distance objective for this dataset and scale. Co-clustering agreement measures repeated-run partition stability. Visible/domain geometry asks whether the groups are substantively meaningful. A low objective is neither a proof of global optimality nor a substitute for the other two audits.'''


### Answer check


In [ ]:
ATOL = 1e-10
RTOL = 1e-8
expected_inertias_p20 = np.array([57.88189830607605,57.881898306076046,57.881898306076046,57.881898306076046,57.881898306076046,57.881898306076046,57.881898306076046,57.881898306076046])
assert audit_p20["inertias"].shape == (8,) and audit_p20["iterations"].shape == (8,)
assert audit_p20["labels"].shape == (8,90) and audit_p20["centers"].shape == (8,3,2)
assert np.allclose(audit_p20["inertias"], expected_inertias_p20, atol=ATOL, rtol=RTOL)
assert audit_p20["best_seed"] == 20260805
assert np.allclose(audit_p20["agreements_to_best"], np.ones(8), atol=ATOL, rtol=RTOL)
assert set(audit_p20) == {"seeds","models","inertias","iterations","labels","centers","co_clustering","best_index","best_seed","agreements_to_best"}
assert isinstance(interpretation_p20, str) and "global" in interpretation_p20
